# 🎵 Feature Extraction Pipeline (MFCC Code Chay)

**Mục tiêu:**
Notebook này thực hiện trích xuất đặc trưng tương tự `feature_extraction.ipynb`, nhưng phần tính toán **MFCC sẽ được thực hiện thủ công bằng Numpy thuần (code chay)** thay vì dùng hàm có sẵn `librosa.feature.mfcc`.
Các bước tự cài đặt:
1. Padding & Chia khung tín hiệu (Framing)
2. Áp dụng cửa sổ (Windowing)
3. Tính phổ năng lượng (FFT & Power Spectrum)
4. Áp dụng bộ lọc Mel (Mel Filterbank)
5. Chuyển sang thang Logarit (Log-Mel)
6. Biến đổi Cosine rời rạc (DCT)


In [ ]:
from pathlib import Path

# ─── Root project (Beta/) ────────────────────────────────────────────────────
PROJECT_ROOT  = Path().resolve().parents[1]   # Beta/

# ─── Đường dẫn dữ liệu ───────────────────────────────────────────────────────
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
FEATURES_DIR  = PROJECT_ROOT / "data" / "features"

# ─── Tham số MFCC ────────────────────────────────────────────────────────────
TARGET_SR  = 16_000
N_MFCC     = 40
N_FFT      = 2048
HOP_LENGTH = 512
N_MELS     = 128

# ─── Tham số Spectral Contrast ───────────────────────────────────────────────
SC_N_BANDS = 6
SC_FMIN    = 200.0

# ─── Tổng chiều embedding ────────────────────────────────────────────────────
EMBEDDING_DIM = N_MFCC * 2 + (SC_N_BANDS + 1) + 12   # = 40+40+7+12 = 99

print(f"PROJECT_ROOT  : {PROJECT_ROOT}")
print(f"PROCESSED_DIR : {PROCESSED_DIR}")
print(f"FEATURES_DIR  : {FEATURES_DIR}")
print(f"EMBEDDING_DIM : {EMBEDDING_DIM}")
assert EMBEDDING_DIM == 99, f"Kiểm tra lại công thức! Nhận được {EMBEDDING_DIM}"


In [ ]:
import json
import logging
from collections import Counter

import numpy as np
import librosa
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)

print(f"librosa version : {librosa.__version__}")
print(f"numpy version   : {np.__version__}")


## Các hàm hỗ trợ tính MFCC bằng Numpy (Code Chay)

In [ ]:
def create_frames_centered(y, n_fft, hop_length):
    """Bước 1: Padding tín hiệu và chia thành các khung (frames)."""
    pad_length = n_fft // 2
    padded_y = np.pad(y, pad_length, mode='reflect')
    
    num_frames = 1 + (len(padded_y) - n_fft) // hop_length
    frames = np.zeros((num_frames, n_fft))
    for i in range(num_frames):
        frames[i] = padded_y[i * hop_length : i * hop_length + n_fft]
    return frames

def apply_window(frames):
    """Bước 2: Áp dụng cửa sổ Hann (Hanning) để làm mịn hai đầu mỗi frame."""
    n_fft = frames.shape[1]
    window = 0.5 - 0.5 * np.cos(2 * np.pi * np.arange(n_fft) / n_fft)
    return frames * window

def compute_power_spectrum(frames):
    """Bước 3: Tính toán Phổ Năng Lượng bằng rFFT."""
    mag_spectrum = np.abs(np.fft.rfft(frames))
    return mag_spectrum ** 2

def hz_to_mel(hz):
    return 2595.0 * np.log10(1.0 + hz / 700.0)

def mel_to_hz(mel):
    return 700.0 * (10.0**(mel / 2595.0) - 1.0)

def get_mel_filterbanks(sr, n_fft, n_mels):
    """Bước 4: Tạo bộ lọc Mel (Mel Filterbank)."""
    low_mel = 0
    high_mel = hz_to_mel(sr / 2.0)
    mel_points = np.linspace(low_mel, high_mel, n_mels + 2)
    hz_points = mel_to_hz(mel_points)
    
    bin_points = np.floor((n_fft + 1) * hz_points / sr).astype(int)
    
    fbank = np.zeros((n_mels, int(np.floor(n_fft / 2 + 1))))
    for m in range(1, n_mels + 1):
        f_m_minus, f_m, f_m_plus = bin_points[m - 1], bin_points[m], bin_points[m + 1]
        
        for k in range(f_m_minus, f_m):
            fbank[m - 1, k] = (k - f_m_minus) / (f_m - f_m_minus)
        for k in range(f_m, f_m_plus):
            fbank[m - 1, k] = (f_m_plus - k) / (f_m_plus - f_m)
            
    return fbank

def apply_dct(log_mel, n_mfcc):
    """Bước 6: Biến đổi Cosine rời rạc (DCT type 2) để thu hệ số MFCC."""
    n_frames, n_mels = log_mel.shape
    mfcc = np.zeros((n_frames, n_mfcc))
    
    for k in range(n_mfcc):
        n = np.arange(n_mels)
        # Hệ số chuẩn hóa Orthogonal
        scale = np.sqrt(1.0 / n_mels) if k == 0 else np.sqrt(2.0 / n_mels)
        
        mfcc[:, k] = scale * np.sum(
            log_mel * np.cos(np.pi * k * (2.0 * n + 1.0) / (2.0 * n_mels)), 
            axis=1
        )
    return mfcc.T  # Đảo lại shape (n_mfcc, n_frames)

def compute_mfcc_scratch(y, sr=16000, n_mfcc=40, n_fft=2048, hop_length=512, n_mels=128):
    """Hàm tổng hợp: Trích xuất MFCC từ tín hiệu âm thanh."""
    frames     = create_frames_centered(y, n_fft, hop_length)
    windowed   = apply_window(frames)
    power_spec = compute_power_spectrum(windowed)
    fbank      = get_mel_filterbanks(sr, n_fft, n_mels)
    
    # Bước 5: Áp dụng filterbank vào power spectrum & Chuyển sang Logarit
    mel_spec = np.dot(power_spec, fbank.T)
    mel_spec = np.maximum(mel_spec, 1e-10) # Tránh log(0)
    log_mel = 10.0 * np.log10(mel_spec)
    
    mfcc = apply_dct(log_mel, n_mfcc)
    return mfcc


## Hàm Trích Xuất và Xây Dựng Embedding

In [ ]:
def extract_features(wav_path: Path) -> dict:
    """
    Trích xuất MFCC (Code Chay), Spectral Contrast, Chroma STFT từ một file .wav.
    """
    # ── 1. Load audio ────────────────────────────────────────────────────────
    y, sr = librosa.load(str(wav_path), sr=TARGET_SR, mono=True)

    # ── 2. MFCC (BẰNG HÀM CODE CHAY NUMPY) ───────────────────────────────────
    mfcc = compute_mfcc_scratch(
        y=y,
        sr=sr,
        n_mfcc=N_MFCC,
        n_fft=N_FFT,
        hop_length=HOP_LENGTH,
        n_mels=N_MELS,
    )
    
    mfcc_mean = np.mean(mfcc, axis=1).astype(np.float32)  # (40,)
    mfcc_std  = np.std(mfcc,  axis=1).astype(np.float32)  # (40,)

    # ── 3. Spectral Contrast ─────────────────────────────────────────────────
    sc = librosa.feature.spectral_contrast(y=y, sr=sr, n_bands=SC_N_BANDS, fmin=SC_FMIN)
    sc_mean = np.mean(sc, axis=1).astype(np.float32)  # (7,)

    # ── 4. Chroma STFT ───────────────────────────────────────────────────────
    chroma = librosa.feature.chroma_stft(y=y, sr=sr, n_fft=N_FFT, hop_length=HOP_LENGTH)
    chroma_mean = np.mean(chroma, axis=1).astype(np.float32)  # (12,)

    return {
        "mfcc_mean"   : mfcc_mean,
        "mfcc_std"    : mfcc_std,
        "sc_mean"     : sc_mean,
        "chroma_mean" : chroma_mean,
        "mfcc_raw"    : mfcc,
    }


In [ ]:
def build_embedding(features: dict) -> np.ndarray:
    """Ghép nối (concatenate) tất cả đặc trưng thành vector 99 chiều."""
    embedding = np.concatenate([
        features["mfcc_mean"],    # [0:40]
        features["mfcc_std"],     # [40:80]
        features["sc_mean"],      # [80:87]
        features["chroma_mean"],  # [87:99]
    ])
    assert embedding.shape == (EMBEDDING_DIM,)
    return embedding


## Chạy Test & Sanity Check

In [ ]:
# So sánh thử kết quả code chay với thư viện Librosa
test_files = sorted(PROCESSED_DIR.rglob("*.wav"))
if test_files:
    test_file = test_files[0]
    y, sr = librosa.load(test_file, sr=TARGET_SR)
    
    mfcc_scratch = compute_mfcc_scratch(y, sr, N_MFCC, N_FFT, HOP_LENGTH, N_MELS)
    mfcc_librosa = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=N_MFCC, n_fft=N_FFT, hop_length=HOP_LENGTH, n_mels=N_MELS)
    
    print("MFCC Scratch shape:", mfcc_scratch.shape)
    print("MFCC Librosa shape:", mfcc_librosa.shape)
    
    # Tính sai số
    mse = np.mean((mfcc_scratch - mfcc_librosa) ** 2)
    print(f"Mean Squared Error (Code chay vs Librosa): {mse:.6f}")
    
    plt.figure(figsize=(10, 4))
    plt.subplot(1, 2, 1)
    plt.title("MFCC (Code Chay)")
    plt.imshow(mfcc_scratch, aspect='auto', origin='lower')
    plt.subplot(1, 2, 2)
    plt.title("MFCC (Librosa)")
    plt.imshow(mfcc_librosa, aspect='auto', origin='lower')
    plt.show()

    feats = extract_features(test_file)
    emb = build_embedding(feats)
    print("Shape vector embedding của file test:", emb.shape)


In [ ]:
def process_all(processed_dir: Path, features_dir: Path) -> list:
    records = []
    errors  = []
    wav_files = sorted(processed_dir.rglob("*.wav"))
    log.info(f"Tìm thấy {len(wav_files)} file .wav để xử lý")

    for wav_path in tqdm(wav_files[:5], desc="Trích xuất đặc trưng (test 5 file)"):
        speaker = wav_path.parent.name
        npy_dir  = features_dir / speaker
        npy_dir.mkdir(parents=True, exist_ok=True)
        npy_path = npy_dir / (wav_path.stem + "_scratch.npy")

        try:
            features  = extract_features(wav_path)
            embedding = build_embedding(features)
            
            # Lưu mảng mfcc raw để debug/test sau này
            np.save(str(npy_path), features["mfcc_raw"])
            
            records.append({
                "speaker"     : speaker,
                "file_path"   : str(wav_path.relative_to(processed_dir.parent)),
                "npy_path"    : str(npy_path.relative_to(features_dir.parent)),
                "sample_rate" : TARGET_SR,
                "duration_sec": 5.0,
                "embedding"   : embedding.tolist(),
            })
        except Exception as e:
            log.warning(f"Lỗi khi xử lý {wav_path.name}: {e}")
            errors.append({"file": str(wav_path), "error": str(e)})

    return records, errors

records, errors = process_all(PROCESSED_DIR, FEATURES_DIR)
if records:
    print(f"Đã xử lý {len(records)} files mẫu bằng code chay.")
